In [ ]:
%pip install agentops langgraph langchain-openai langchain-tavily python-dotenv


In [1]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import agentops
from agentops import (
    init,
    start_trace,
    end_trace,
    agent,
    operation,
    tool,
)
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import logging


In [2]:
import os
from dotenv import load_dotenv

# Carregar o arquivo .env
load_dotenv(
    r"C:\Users\eugen\Documents\ProjetoIA\.env",
    override=True
)

# Verificar configurações
print("OpenAI:", bool(os.getenv("OPENAI_API_KEY")))
print("Tavily:", bool(os.getenv("TAVILY_API_KEY")))
print("AgentOps:", bool(os.getenv("AGENTOPS_API_KEY")))

OpenAI: True
Tavily: True
AgentOps: True


In [3]:
# Corrigir o erro de codificação
original_factory = logging.getLogRecordFactory()

def safe_record_factory(*args, **kwargs):
    record = original_factory(*args, **kwargs)
    if isinstance(record.msg, str) and "🤔" in record.msg:
        record.msg = record.msg.replace("🤔", "")
    return record

logging.setLogRecordFactory(safe_record_factory)





In [4]:
import agentops
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv(
    r"C:\Users\eugen\Documents\ProjetoIA\.env",
    override=True
)
agentops.init(api_key=os.environ['AGENTOPS_API_KEY'], default_tags=["langgraph-agent"])

🖇 AgentOps: You're on the agentops free plan 


In [57]:
class AgentState(TypedDict):
    query: str
    search_results: str
    summary: str
    final_answer: str

In [58]:
# Inicializar o modelo da OpenAI
model = ChatOpenAI(model_name="gpt-4o", temperature=0)

# Inicializar o Tavily Search
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(max_results=3)

In [59]:

from langchain_tavily import TavilySearch
from agentops.sdk.decorators import tool as agentops_tool

@agentops_tool(name="web_search")
def web_search(state: AgentState) -> AgentState:
    """Realiza buscas na internet utilizando Tavily."""

    try:
        # Inicializar Tavily
        tavily_search = TavilySearch(
            max_results=3,
            search_depth="advanced"
        )

        # Realizar pesquisa
        search_results = tavily_search.invoke({
            "query": state["query"]
        })

        # Extrair resultados
        if isinstance(search_results, dict):
            results = search_results.get("results", [])
        elif isinstance(search_results, list):
            results = search_results
        else:
            results = [search_results]

        # Formatar resultados
        formatted_results = "\n".join(
            f"{r.get('title', '')}: {r.get('content', '')}"
            if isinstance(r, dict) else str(r)
            for r in results
        )

        state["search_results"] = (
            formatted_results or "Nenhum resultado encontrado."
        )

    except Exception as e:
        print(f"Erro no Tavily: {e}")
        raise

    return state


In [60]:

from agentops.sdk.decorators import operation
from langchain_core.prompts import ChatPromptTemplate

# Registrar operação no AgentOps
@operation(name="summarize_results")
def summarize_results(state: AgentState) -> AgentState:
    """Resume os resultados da pesquisa utilizando LLM."""

    # Criar prompt
    prompt = ChatPromptTemplate.from_template(
        "Resuma as seguintes informações em um parágrafo conciso em português.\n\nInformações: {search_results}"
        
    )

    # Criar cadeia de processamento
    chain = prompt | model

    # Executar modelo
    summary = chain.invoke({
        "search_results": state["search_results"]
    }).content

    # Atualizar estado
    state["summary"] = summary

    return state


In [61]:
from typing import TypedDict


from agentops.sdk.decorators import operation
from langchain_core.prompts import ChatPromptTemplate

@operation(name="generate_answer")
def generate_answer(state: AgentState) -> AgentState:
    prompt = ChatPromptTemplate.from_template(
        "Com base na sua pergunta '{query}' e nas informações resumidas abaixo, forneça uma resposta detalhada em português, utilizando bullet points quando necessário.\n\nResumo: {summary}"
    )
    chain = prompt | model
    final_answer = chain.invoke({"query": state["query"], "summary": state["summary"]}).content
    state["final_answer"] = final_answer
    return state

In [62]:
from langgraph.graph import StateGraph, START, END
# Construção do grafo
builder = StateGraph(AgentState)

# Adicionar nós
builder.add_node("web_search", web_search)
builder.add_node("summarize_results", summarize_results)
builder.add_node("generate_answer", generate_answer)

# Adicionar arestas
builder.add_edge(START, "web_search")
builder.add_edge("web_search", "summarize_results")
builder.add_edge("summarize_results", "generate_answer")
builder.add_edge("generate_answer", END)



# Compilar o grafo
graph = builder.compile()

In [63]:
from agentops.sdk.decorators import trace

@trace(name="langgraph-workflow", tags=["langgraph-agent"])
def execute_workflow(query: str) -> str:
    """Executa o LangGraph com rastreamento pelo AgentOps."""

    initial_state = {
        "query": query,
        "search_results": "",
        "summary": "",
        "final_answer": ""
    }

    final_state = graph.invoke(initial_state)

    print("Campos retornados:", list(final_state.keys()))

    answer = final_state.get("final_answer")

    if not answer:
        raise RuntimeError(
            "O nó generate_answer não produziu uma resposta final."
        )

    return answer

In [66]:


# Exemplo de consulta
query = "liste todos os cursos da scoras academy?"

# Executar o fluxo de trabalho
answer = execute_workflow(query)

# Exibir a resposta final
print("Resposta Final:\n")
print(answer)

🖇 AgentOps: Session Replay for langgraph-workflow trace: https://app.agentops.ai/sessions?trace_id=d0fbf446126543b5a6a2a957b08f5598


Campos retornados: ['query', 'search_results', 'summary', 'final_answer']


🖇 AgentOps: Session Replay for langgraph-workflow.session trace: https://app.agentops.ai/sessions?trace_id=d0fbf446126543b5a6a2a957b08f5598


Resposta Final:

A Scoras Academy, sob a liderança de Anderson Amaral, oferece uma formação abrangente em agentes de IA. A seguir, estão listados os cursos e tópicos cobertos pela academia:

- **Fundamentos de LLMs e SLMs**: 
  - Introdução aos modelos de linguagem de larga escala (LLMs) e modelos de linguagem de pequena escala (SLMs).
  
- **Model Context Protocol**:
  - Abordagem para estruturar e gerenciar o contexto em que os modelos de IA operam.

- **Frameworks**:
  - **PydanticAI**: Ferramenta para validação de dados e gerenciamento de configurações em projetos de IA.
  - **LangGraph**: Framework para construção de fluxos de trabalho em IA.

- **Construção de Agentes de IA**:
  - Uso de tecnologias como DuckDB e MongoDB para desenvolvimento de agentes inteligentes.

- **Aplicação Prática de IA em Ambientes Corporativos**:
  - Projetos práticos que incluem:
    - Automação de processos.
    - Análise de contratos.
    - Análise de dados climáticos.

- **Trilhas de Aprendizado**:
